<a href="https://colab.research.google.com/github/Leiva230-2/Research-Methodology-Code/blob/main/RM_Code_COLAB_FIXED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parking Occupancy — Compression Pipeline (Colab, fixed)

## Why the old notebook kept breaking

`tensorflow_model_optimization` (QAT + pruning) needs **Keras 2**. Modern Colab
ships **Keras 3**. The bridge is:

1. the `tf-keras` package must be installed, **and**
2. `TF_USE_LEGACY_KERAS=1` must be set **before TensorFlow is imported anywhere**.

In the old notebook that flag was set in cells 2, 3 and 11 — so whether it worked
depended on which cell you ran first after a runtime restart. That is the "random"
failure. Here it is set once, in the first cell after install, and the runtime is
restarted deliberately.

## Run order

* **Cell 1** → installs, then the runtime restarts itself. This is expected.
* **Cell 2 onward** → run top to bottom.

Everything is written to Google Drive, so a disconnect no longer costs you a
full retrain — completed stages are skipped on rerun.

## Step 1 — Install (runtime will restart automatically)

In [ ]:
# Pinned so tfmot and Keras 2 agree with each other.
!pip install -q tensorflow-model-optimization==0.8.0 tf-keras==2.18.0 psutil "opencv-python-headless==4.10.0.84"

print("\nInstall done. Restarting runtime — this is expected, not an error.")
print("After it restarts, continue from Step 2. Do NOT rerun this cell.")

import IPython
IPython.get_ipython().kernel.do_shutdown(True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 35.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.6/615.6 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 71.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-text 2.20.1 requires tensorflow<2.21,>=2.20.0, but you have tensorflow 2.18.1 which is incompatible.
rasterio 1.5.1 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.


{'status': 'ok', 'restart': True}

In [ ]:
# ## FIX

# import os
# os.environ["TF_USE_LEGACY_KERAS"] = "1"

# import tensorflow as tf
# import tensorflow_model_optimization as tfmot

# print("tf.keras ->", tf.keras.__name__)
# assert "tf_keras" in tf.keras.__name__, (
#     f"WRONG: {tf.keras.__name__}. Restart runtime and run THIS cell first."
# )
# print("Keras 2 active ✓")

tf.keras -> tf_keras.api._v2.keras
Keras 2 active ✓


## Step 2 — Environment setup

Run this **first** after the restart. It sets the legacy-Keras flag before
TensorFlow is imported, mounts Drive, and verifies the GPU.

In [ ]:
# ── The flag MUST be set before any tensorflow import ────────────────────
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
import tensorflow_model_optimization as tfmot
import keras

print("TensorFlow :", tf.__version__)
print("tfmot      :", tfmot.__version__)

keras_module = tf.keras.__name__
print("tf.keras   :", keras_module)
assert "tf_keras" in keras_module, (
    f"tf.keras resolved to '{keras_module}', expected tf_keras (Keras 2)."
)
print("Keras 2 confirmed — tfmot will work.\n")

import cv2, numpy
print("numpy      :", numpy.__version__)
print("cv2        :", cv2.__version__, "- OK")

gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus if gpus else "NONE — Runtime > Change runtime type > T4 GPU")

TensorFlow : 2.18.1
tfmot      : 0.8.0
tf.keras   : tf_keras.api._v2.keras
Keras 2 confirmed — tfmot will work.

numpy      : 1.26.4
cv2        : 4.10.0 - OK
GPU: NONE — Runtime > Change runtime type > T4 GPU


In [ ]:
import random
from tf_keras.src import backend as _k_backend

def _patched_create_seed(self, user_specified_seed):
    if user_specified_seed is not None:
        return user_specified_seed
    return random.randint(1, int(1e9))

_k_backend.RandomGenerator._create_seed = _patched_create_seed
print("Patched Keras _create_seed for Python 3.12")

Patched Keras _create_seed for Python 3.12


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

# Everything that must survive a disconnect lives here
DRIVE_ROOT = Path('/content/drive/MyDrive/parking_compression')
MODEL_DIR  = DRIVE_ROOT / 'models'
TFLITE_DIR = DRIVE_ROOT / 'tflite'
RESULTS_DIR= DRIVE_ROOT / 'results'

# Fast scratch space (wiped on disconnect — that is fine, it is regenerable)
DATA_DIR   = Path('/content/dataset_processed')
PKLOT_DIR  = Path('/content/pklot/PKLot/PKLot')

for d in (MODEL_DIR, TFLITE_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Experiment settings ──────────────────────────────────────────────────
IMG_SIZE = (224, 224)
BATCH    = 32
EPOCHS_HEAD, EPOCHS_FINE = 20, 30
LR_HEAD, LR_FINE = 1e-3, 1e-4
SPLIT_SEED = 42
MAX_PER_CLASS = 2000
WEATHER_MAP = {'sunny': 'sunny', 'cloudy': 'overcast', 'rainy': 'rainy'}

PRUNING_LEVELS = [0.10, 0.20, 0.30, 0.40, 0.50]   # Reviewer 2
PRUNING_EPOCHS = 3
SEEDS = [0, 1, 2]                                  # Reviewer 2/3: CIs
QAT_LEARNING_RATES = [1e-5, 5e-5, 1e-4]            # Reviewer 2: QAT ablation
QAT_EPOCHS_GRID = [3, 10]

BENCH_WARMUP, BENCH_RUNS = 10, 1000

print("Drive root:", DRIVE_ROOT)

Mounted at /content/drive
Drive root: /content/drive/MyDrive/parking_compression


## Step 3 — Dataset

Downloads PKLot and builds the cropped dataset. If a prepared copy already
exists on Drive it is restored instead of rebuilt (saves ~15 min per session).

In [ ]:
!lscpu | grep -E "Model name|^CPU\(s\)"
!cat /etc/os-release | head -2
import sys, tensorflow as tf
print("Python:", sys.version.split()[0], "| TF:", tf.__version__)

CPU(s):                                  2
Model name:                              AMD EPYC 7B12
PRETTY_NAME="Ubuntu 22.04.5 LTS"
NAME="Ubuntu"
Python: 3.12.13 | TF: 2.18.1


In [ ]:
# Kaggle credentials: upload kaggle.json (Kaggle > Account > Create New API Token)
import os, json

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": "Aviel Aquino", "key": "KGAT_f64380cce4ba38a07b2c06e49de4c87e"}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print("kaggle.json ready")

kaggle.json ready


In [ ]:
import shutil, os
from pathlib import Path

DATASET_ARCHIVE = DRIVE_ROOT / 'dataset_processed.zip'

if DATASET_ARCHIVE.exists():
    print("Found prepared dataset on Drive — restoring instead of rebuilding.")
    shutil.unpack_archive(str(DATASET_ARCHIVE), '/content/')
    print("Restored ->", DATA_DIR)
elif not PKLOT_DIR.exists():
    print("Downloading PKLot (a few minutes)...")
    import subprocess
    os.makedirs('/content/pklot', exist_ok=True)
    subprocess.run(
        ["kaggle", "datasets", "download", "--unzip",
         "-d", "blanderbuss/parking-lot-dataset"],
        cwd="/content/pklot", check=True)
    print("PKLot ready:", PKLOT_DIR.exists())
else:
    print("PKLot already present.")

Found prepared dataset on Drive — restoring instead of rebuilding.
Restored -> /content/dataset_processed


In [ ]:
import os, cv2, random
import xml.etree.ElementTree as ET
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from collections import Counter, defaultdict

random.seed(SPLIT_SEED)

def parse_pklot():
    all_data = []
    for root_dir, _, files in os.walk(str(PKLOT_DIR)):
        for f in files:
            if not f.endswith('.xml'): continue
            xml_path = os.path.join(root_dir, f)
            img_path = xml_path.replace('.xml', '.jpg')
            if not os.path.exists(img_path): continue
            weather_raw = next((w for w in ['sunny','cloudy','rainy']
                                if w in root_dir.lower()), None)
            if not weather_raw: continue
            weather = WEATHER_MAP[weather_raw]
            try:
                xroot = ET.parse(xml_path).getroot()
            except Exception:
                continue
            for space in xroot.findall('space'):
                label = 'occupied' if space.get('occupied') == '1' else 'empty'
                contour = space.find('contour')
                if contour is None: continue
                pts = contour.findall('point')
                if len(pts) < 3: continue
                xs = [int(p.get('x')) for p in pts]
                ys = [int(p.get('y')) for p in pts]
                all_data.append({'img_path': img_path,
                                 'bbox': (min(xs),min(ys),max(xs),max(ys)),
                                 'label': label, 'weather': weather})
    print(f"Total spaces: {len(all_data)}")
    print("Weather:", Counter(d['weather'] for d in all_data))
    print("Labels :", Counter(d['label']   for d in all_data))
    return all_data

def stratified_sample(data, max_per_class=MAX_PER_CLASS):
    buckets = defaultdict(list)
    for item in data:
        buckets[f"{item['weather']}_{item['label']}"].append(item)
    sampled = []
    for _, items in buckets.items():
        random.shuffle(items)
        sampled.extend(items[:max_per_class])
    random.shuffle(sampled)
    print(f"Sampled {len(sampled)} from {len(data)}")
    return sampled

def stratified_split(data):
    strat = [f"{d['weather']}_{d['label']}" for d in data]
    train, tmp = train_test_split(data, test_size=0.30, stratify=strat,
                                  random_state=SPLIT_SEED)
    strat2 = [f"{d['weather']}_{d['label']}" for d in tmp]
    val, test = train_test_split(tmp, test_size=0.50, stratify=strat2,
                                 random_state=SPLIT_SEED)
    print(f"Train {len(train)} | Val {len(val)} | Test {len(test)}")
    return train, val, test

def crop_and_save(dataset, split):
    saved = skipped = 0
    for idx, item in enumerate(tqdm(dataset, desc=split)):
        out_dir = DATA_DIR / split / item['weather'] / item['label']
        out_dir.mkdir(parents=True, exist_ok=True)
        img = cv2.imread(item['img_path'])
        if img is None: skipped += 1; continue
        h, w = img.shape[:2]
        x1,y1,x2,y2 = item['bbox']
        x1,x2,y1,y2 = max(0,x1),min(w,x2),max(0,y1),min(h,y2)
        if x2<=x1 or y2<=y1: skipped += 1; continue
        crop = img[y1:y2, x1:x2]
        if crop.size == 0: skipped += 1; continue
        cv2.imwrite(str(out_dir / f"{split}_{item['weather']}_{idx:06d}.jpg"),
                    cv2.resize(crop, IMG_SIZE))
        saved += 1
    print(f"  saved={saved}  skipped={skipped}")

if (DATA_DIR / 'train').exists():
    print("Dataset already prepared this session — skipping.")
else:
    data = stratified_sample(parse_pklot())
    train_d, val_d, test_d = stratified_split(data)
    for split, d in [('train',train_d), ('valid',val_d), ('test',test_d)]:
        crop_and_save(d, split)

    # Archive to Drive so future sessions restore in seconds
    print("\nArchiving to Drive for future sessions...")
    shutil.make_archive(str(DRIVE_ROOT / 'dataset_processed'), 'zip',
                        '/content', 'dataset_processed')
    print("Saved ->", DATASET_ARCHIVE)

Dataset already prepared this session — skipping.


## Step 4 — Data loaders

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV3Large
import numpy as np, os

def make_dataset(split, augment=False, seed=SPLIT_SEED):
    paths, labels = [], []
    split_dir = DATA_DIR / split
    for weather in sorted(os.listdir(split_dir)):
        for lbl in ['empty','occupied']:
            folder = split_dir / weather / lbl
            if not folder.is_dir(): continue
            for f in sorted(os.listdir(folder)):
                if f.endswith('.jpg'):
                    paths.append(str(folder / f))
                    labels.append(1 if lbl == 'occupied' else 0)

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load(p, y):
        img = tf.io.read_file(p)
        img = tf.image.decode_jpeg(img, channels=3)
        img = tf.image.resize(img, IMG_SIZE)
        mean = tf.constant([123.68, 116.78, 103.94])
        std  = tf.constant([58.8, 58.8, 58.8])
        return (img - mean) / std, y

    def augment_fn(img, y):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.2)
        img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
        return img, y

    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.shuffle(1000, seed=seed).batch(BATCH).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset('train', augment=True)
val_ds   = make_dataset('valid')
print("train batches:", len(train_ds), "| val batches:", len(val_ds))

train batches: 263 | val batches: 57


## Step 5 — FP32 baseline

Saved to Drive. Reruns skip training if the file already exists — delete it on
Drive if you want a genuine retrain.

In [ ]:
import tensorflow as tf; print(tf.keras.__name__)

tensorflow.keras


In [ ]:
def build_and_train_fp32(seed=0, save_name="mobilenetv3_fp32.keras",
                         verbose=1, force=False):
    out_path = MODEL_DIR / save_name
    if out_path.exists() and not force:
        print(f"{save_name} already on Drive — loading (force=True to retrain).")
        return tf.keras.models.load_model(str(out_path))

    tf.keras.utils.set_random_seed(seed)

    base = MobileNetV3Large(weights='imagenet', include_top=False,
                            input_shape=(*IMG_SIZE, 3))
    base.trainable = False
    x = base.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(2, activation='softmax')(x)
    model = Model(base.input, out)

    model.compile(optimizer=tf.keras.optimizers.Adam(LR_HEAD),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, verbose=verbose,
              callbacks=[tf.keras.callbacks.EarlyStopping(patience=5,
                                                          restore_best_weights=True)])

    base.trainable = True
    model.compile(optimizer=tf.keras.optimizers.Adam(LR_FINE),
                  loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FINE, verbose=verbose,
              callbacks=[tf.keras.callbacks.EarlyStopping(patience=7,
                                                          restore_best_weights=True)])

    model.save(str(out_path))
    print("Saved ->", out_path)
    return model

fp32_model = build_and_train_fp32(seed=0)

mobilenetv3_fp32.keras already on Drive — loading (force=True to retrain).


## Step 6 — Dynamic-range PTQ

In [ ]:
def export_dynamic_ptq(keras_path, out_name):
    m = tf.keras.models.load_model(str(keras_path))
    conv = tf.lite.TFLiteConverter.from_keras_model(m)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    blob = conv.convert()
    out = TFLITE_DIR / out_name
    out.write_bytes(blob)
    print(f"{out_name}: {len(blob)/1e6:.2f} MB")
    return out

export_dynamic_ptq(MODEL_DIR / "mobilenetv3_fp32.keras",
                   "mobilenetv3_ptq_dynamic.tflite")

mobilenetv3_ptq_dynamic.tflite: 3.27 MB


PosixPath('/content/drive/MyDrive/parking_compression/tflite/mobilenetv3_ptq_dynamic.tflite')

## Step 7 — QAT

`train_qat` is parameterised so Reviewer 2's ablation (learning rate,
fine-tuning duration, BN handling) is a loop rather than a manual edit.

In [ ]:
# def train_qat_v2(lr=1e-5, epochs=3, freeze_bn=True, seed=0, tag=None,
#                  full_model_quantize=True, verbose=1):
#     """QAT with diagnostics: saves the Keras model and exports two TFLite variants."""
#     tf.keras.utils.set_random_seed(seed)
#     m = tf.keras.models.load_model(str(MODEL_DIR / "mobilenetv3_fp32.keras"))

#     if full_model_quantize:
#         # Whole-model quantization — handles hard-swish / SE blocks coherently
#         qat_model = tfmot.quantization.keras.quantize_model(m)
#     else:
#         def annotate(layer):
#             target = (tf.keras.layers.Conv2D, tf.keras.layers.Dense,
#                       tf.keras.layers.DepthwiseConv2D)
#             return (tfmot.quantization.keras.quantize_annotate_layer(layer)
#                     if isinstance(layer, target) else layer)
#         annotated = tf.keras.models.clone_model(m, clone_function=annotate)
#         with tfmot.quantization.keras.quantize_scope():
#             qat_model = tfmot.quantization.keras.quantize_apply(annotated)

#     if freeze_bn:
#         for layer in qat_model.layers:
#             if isinstance(layer, tf.keras.layers.BatchNormalization):
#                 layer.trainable = False

#     qat_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
#                       loss='sparse_categorical_crossentropy', metrics=['accuracy'])
#     hist = qat_model.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=verbose)

#     tag = tag or f"lr{lr:g}_ep{epochs}_{'full' if full_model_quantize else 'sel'}"

#     # 1. Save the Keras model — lets us test whether conversion is the culprit
#     keras_out = MODEL_DIR / f"mobilenetv3_qat_{tag}.keras"
#     qat_model.save(str(keras_out))

#     # 2. Export with INT8 optimization (the standard QAT recipe)
#     conv = tf.lite.TFLiteConverter.from_keras_model(qat_model)
#     conv.optimizations = [tf.lite.Optimize.DEFAULT]
#     int8_out = TFLITE_DIR / f"mobilenetv3_qat_{tag}_int8.tflite"
#     int8_out.write_bytes(conv.convert())

#     # 3. Export WITHOUT optimization — isolates double-quantization as a cause
#     conv2 = tf.lite.TFLiteConverter.from_keras_model(qat_model)
#     conv2.optimizations = []
#     float_out = TFLITE_DIR / f"mobilenetv3_qat_{tag}_float.tflite"
#     float_out.write_bytes(conv2.convert())

#     print(f"\nsaved: {keras_out.name}, {int8_out.name}, {float_out.name}")

#     # 4. Immediate three-way comparison
#     print("\n--- Keras QAT model ---")
#     a_keras = eval_per_weather(keras_out)
#     print("--- TFLite INT8 ---")
#     a_int8 = eval_per_weather(int8_out, is_tflite=True)
#     print("--- TFLite float ---")
#     a_float = eval_per_weather(float_out, is_tflite=True)

#     return {"tag": tag, "val_acc": float(hist.history["val_accuracy"][-1]),
#             "keras": a_keras["overall"], "int8": a_int8["overall"],
#             "float": a_float["overall"]}

# r = train_qat_v2(lr=1e-4, epochs=10, full_model_quantize=True, tag="fixed")
# print(r)

In [ ]:
import numpy as np, os, json

def eval_model_obj(model, is_tflite_path=None):
    """Evaluate an in-memory Keras model (or a tflite path) on the test set."""
    mean = np.array([123.68,116.78,103.94], dtype=np.float32); std = np.float32(58.8)
    if is_tflite_path:
        interp = tf.lite.Interpreter(model_path=str(is_tflite_path))
        interp.allocate_tensors()
        inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
    res = {}
    tot_c = tot_n = 0
    for weather in ['sunny','overcast','rainy']:
        c = n = 0
        for lbl in ['empty','occupied']:
            folder = DATA_DIR/"test"/weather/lbl
            if not folder.is_dir(): continue
            y = 1 if lbl=='occupied' else 0
            for f in sorted(os.listdir(folder)):
                if not f.endswith('.jpg'): continue
                img = tf.image.decode_jpeg(tf.io.read_file(str(folder/f)), channels=3)
                x = ((tf.image.resize(img,(224,224)).numpy()-mean)/std)[None].astype(np.float32)
                if is_tflite_path:
                    interp.set_tensor(inp['index'], x); interp.invoke()
                    p = int(np.argmax(interp.get_tensor(out['index'])))
                else:
                    p = int(np.argmax(model(x, training=False).numpy()))
                c += int(p==y); n += 1
        res[weather] = 100*c/n if n else float('nan')
        print(f"  {weather:9s}: {res[weather]:6.2f}%  ({c}/{n})")
        tot_c += c; tot_n += n
    res['overall'] = 100*tot_c/tot_n
    print(f"  {'OVERALL':9s}: {res['overall']:6.2f}%  ({tot_c}/{tot_n})")
    return res

def train_qat_final(lr=1e-4, epochs=10, seed=0, tag="final"):
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.models.load_model(str(MODEL_DIR / "mobilenetv3_fp32.keras"))

    def annotate(layer):
        target = (tf.keras.layers.Conv2D, tf.keras.layers.Dense,
                  tf.keras.layers.DepthwiseConv2D)
        return (tfmot.quantization.keras.quantize_annotate_layer(layer)
                if isinstance(layer, target) else layer)
    annotated = tf.keras.models.clone_model(m, clone_function=annotate)
    with tfmot.quantization.keras.quantize_scope():
        qat = tfmot.quantization.keras.quantize_apply(annotated)

    for l in qat.layers:
        if isinstance(l, tf.keras.layers.BatchNormalization): l.trainable = False

    qat.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=5e-5, clipnorm=1.0),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    hist = qat.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=1)

    qat.save_weights(str(MODEL_DIR / f"qat_{tag}.weights.h5"))   # h5 round-trips

    out = {"val_acc": float(hist.history['val_accuracy'][-1])*100,
           "history": [float(v)*100 for v in hist.history['val_accuracy']]}

    print("\n--- Keras QAT (in memory) ---")
    out["keras_test"] = eval_model_obj(qat)["overall"]

    for vname, opts in [("int8", [tf.lite.Optimize.DEFAULT]), ("noopt", [])]:
        try:
            conv = tf.lite.TFLiteConverter.from_keras_model(qat)
            conv.optimizations = opts
            blob = conv.convert()
            p = TFLITE_DIR / f"mobilenetv3_qat_{tag}_{vname}.tflite"
            p.write_bytes(blob)
            print(f"\n--- TFLite [{vname}] {len(blob)/1e6:.2f} MB ---")
            out[f"tflite_{vname}"] = eval_model_obj(None, is_tflite_path=p)["overall"]
            out[f"size_{vname}_mb"] = len(blob)/1e6
        except Exception as e:
            print(f"[{vname}] failed: {type(e).__name__}: {e}")
            out[f"tflite_{vname}"] = None

    (RESULTS_DIR / f"qat_{tag}.json").write_text(json.dumps(out, indent=2))
    print("\n" + "="*40); print(json.dumps(out, indent=2))
    return out


res = train_qat_final()

Epoch 1/10
263/263 [==============================] - 144s 445ms/step - loss: 0.6406 - accuracy: 0.6707 - val_loss: 0.8160 - val_accuracy: 0.5000
Epoch 2/10
263/263 [==============================] - 115s 430ms/step - loss: 0.5475 - accuracy: 0.7360 - val_loss: 0.3457 - val_accuracy: 0.8239
Epoch 3/10
263/263 [==============================] - 119s 438ms/step - loss: 0.3073 - accuracy: 0.8833 - val_loss: 0.2684 - val_accuracy: 0.8833
Epoch 4/10
263/263 [==============================] - 116s 432ms/step - loss: 0.1783 - accuracy: 0.9458 - val_loss: 0.1474 - val_accuracy: 0.9550
Epoch 5/10
263/263 [==============================] - 117s 432ms/step - loss: 0.1109 - accuracy: 0.9685 - val_loss: 0.1386 - val_accuracy: 0.9606
Epoch 6/10
263/263 [==============================] - 117s 438ms/step - loss: 0.1108 - accuracy: 0.9637 - val_loss: 0.1470 - val_accuracy: 0.9644
Epoch 7/10
263/263 [==============================] - 116s 431ms/step - loss: 0.0948 - accuracy: 0.9721 - val_loss: 0.1883 -

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(



--- TFLite [int8] 6.28 MB ---


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:451: UserWarning:     Warning: Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  sunny    :  59.67%  (358/600)
  overcast :  64.83%  (389/600)
  rainy    :  70.17%  (421/600)
  OVERALL  :  64.89%  (1168/1800)

--- TFLite [noopt] 12.26 MB ---


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:451: UserWarning:     Warning: Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  sunny    :  95.50%  (573/600)
  overcast :  99.17%  (595/600)
  rainy    :  99.17%  (595/600)
  OVERALL  :  97.94%  (1763/1800)

{
  "val_acc": 96.66666388511658,
  "history": [
    50.0,
    82.3888897895813,
    88.33333253860474,
    95.49999833106995,
    96.05555534362793,
    96.44444584846497,
    93.99999976158142,
    96.88888788223267,
    97.2777783870697,
    96.66666388511658
  ],
  "keras_test": 98.05555555555556,
  "tflite_int8": 64.88888888888889,
  "size_int8_mb": 6.27676,
  "tflite_noopt": 97.94444444444444,
  "size_noopt_mb": 12.25588
}


In [ ]:
# def train_qat(lr=1e-5, epochs=3, freeze_bn=True, seed=0, tag=None, verbose=1):
#     tf.keras.utils.set_random_seed(seed)
#     m = tf.keras.models.load_model(str(MODEL_DIR / "mobilenetv3_fp32.keras"))

#     def annotate(layer):
#         target = (tf.keras.layers.Conv2D, tf.keras.layers.Dense,
#                   tf.keras.layers.DepthwiseConv2D)
#         if isinstance(layer, target):
#             return tfmot.quantization.keras.quantize_annotate_layer(layer)
#         return layer

#     annotated = tf.keras.models.clone_model(m, clone_function=annotate)
#     with tfmot.quantization.keras.quantize_scope():
#         qat_model = tfmot.quantization.keras.quantize_apply(annotated)

#     if freeze_bn:
#         for layer in qat_model.layers:
#             if isinstance(layer, tf.keras.layers.BatchNormalization):
#                 layer.trainable = False

#     qat_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
#                       loss='sparse_categorical_crossentropy', metrics=['accuracy'])
#     hist = qat_model.fit(train_ds, validation_data=val_ds,
#                          epochs=epochs, verbose=verbose)

#     conv = tf.lite.TFLiteConverter.from_keras_model(qat_model)
#     conv.optimizations = [tf.lite.Optimize.DEFAULT]
#     blob = conv.convert()

#     tag = tag or f"lr{lr:g}_ep{epochs}_bn{'frozen' if freeze_bn else 'live'}"
#     out = TFLITE_DIR / f"mobilenetv3_qat_{tag}.tflite"
#     out.write_bytes(blob)
#     print(f"QAT -> {out.name} ({len(blob)/1e6:.2f} MB)")
#     return out, hist

# # qat_path, qat_hist = train_qat(lr=1e-5, epochs=3, tag="baseline")

In [ ]:
# ── QAT ablation (Reviewer 2) — slow, run when the rest works ────────────
import json
qat_ablation = []
for lr in QAT_LEARNING_RATES:
    for ep in QAT_EPOCHS_GRID:
        path, hist = train_qat(lr=lr, epochs=ep, verbose=0)
        rec = {"lr": lr, "epochs": ep, "tflite": path.name,
               "final_val_acc": float(hist.history["val_accuracy"][-1])}
        qat_ablation.append(rec)
        print(f"  lr={lr:g} ep={ep} -> val_acc={rec['final_val_acc']:.4f}")

(RESULTS_DIR / "qat_ablation.json").write_text(json.dumps(qat_ablation, indent=2))
print("\nSaved -> results/qat_ablation.json")

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


QAT -> mobilenetv3_qat_lr1e-05_ep3_bnfrozen.tflite (6.28 MB)
  lr=1e-05 ep=3 -> val_acc=0.8328


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


QAT -> mobilenetv3_qat_lr1e-05_ep10_bnfrozen.tflite (6.28 MB)
  lr=1e-05 ep=10 -> val_acc=0.9722


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


QAT -> mobilenetv3_qat_lr5e-05_ep3_bnfrozen.tflite (6.28 MB)
  lr=5e-05 ep=3 -> val_acc=0.6317


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


QAT -> mobilenetv3_qat_lr5e-05_ep10_bnfrozen.tflite (6.28 MB)
  lr=5e-05 ep=10 -> val_acc=0.9672


In [ ]:
#!/usr/bin/env python3
"""
Figure 2 — weather-specific accuracy comparison.

Run locally or in Colab:
    python3 make_figure2.py

Outputs figure2_weather_accuracy.png (300 dpi) and .pdf (vector, preferred
for camera-ready submission).
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ── Data ──────────────────────────────────────────────────────────────────
# Update these if numbers change. Order: sunny, overcast, rainy.
WEATHER = ["Sunny", "Overcast", "Rainy"]

MODELS = {
    "FP32 baseline":  [93.00, 98.33, 98.17],
    "Dynamic PTQ":    [95.00, 98.00, 97.33],
    # Uncomment to include pruning in the comparison:
    # "Pruned 30%":   [93.83, 99.17, 98.50],
    # "Pruned 50%":   [87.33, 87.83, 90.50],
}

# Grayscale-safe: differs in both hue and lightness, prints legibly in B&W
COLORS = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]

# ── Style ─────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 9,
    "legend.fontsize": 8,
    "xtick.labelsize": 9,
    "ytick.labelsize": 8,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

n_models = len(MODELS)
x = np.arange(len(WEATHER))
width = 0.8 / n_models

# IEEE single column is ~3.5in wide
fig, ax = plt.subplots(figsize=(6, 3), dpi=300)

for i, (name, vals) in enumerate(MODELS.items()):
    offset = (i - (n_models - 1) / 2) * width
    bars = ax.bar(x + offset, vals, width * 0.9,
                  label=name, color=COLORS[i % len(COLORS)],
                  edgecolor="black", linewidth=0.4)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.25, f"{v:.1f}",
                ha="center", va="bottom", fontsize=6.5)

# Y-axis is truncated to make differences visible. State this in the caption.
lo = min(min(v) for v in MODELS.values())
ax.set_ylim(max(0, lo - 4), 101)

ax.set_ylabel("Accuracy (%)")
ax.set_xticks(x)
ax.set_xticklabels(WEATHER)
ax.legend(frameon=False, loc="upper right", ncol=1)
ax.grid(axis="y", linestyle=":", linewidth=0.5, alpha=0.6)
ax.set_axisbelow(True)

fig.tight_layout(pad=0.3)
fig.savefig("figure2_weather_accuracy.png", bbox_inches="tight", dpi=300)
fig.savefig("figure2_weather_accuracy.pdf", bbox_inches="tight")
print("Saved figure2_weather_accuracy.{png,pdf}")

# ── Weather Sensitivity Index (std dev across conditions) ────────────────
print("\nWSI (population std dev across weather conditions):")
for name, vals in MODELS.items():
    print(f"  {name:<16} {np.std(vals, ddof=0):.3f}")

Saved figure2_weather_accuracy.{png,pdf}

WSI (population std dev across weather conditions):
  FP32 baseline    2.476
  Dynamic PTQ      1.286


## Step 8 — Pruning at 10/20/30/40/50%

Extended from the original `[0.30, 0.50]`. Existing levels are skipped so an
interrupted session resumes where it stopped.

In [ ]:
def train_pruned(sparsity, epochs=PRUNING_EPOCHS, seed=0, verbose=1, force=False):
    pct = int(round(sparsity * 100))
    out_path = MODEL_DIR / f"mobilenetv3_pruned_{pct}.keras"
    if out_path.exists() and not force:
        print(f"pruned {pct}% already on Drive — skipping.")
        return out_path

    tf.keras.utils.set_random_seed(seed)
    schedule = tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0, final_sparsity=sparsity,
        begin_step=0, end_step=len(train_ds) * epochs)

    def apply_pruning(layer):
        target = (tf.keras.layers.Conv2D, tf.keras.layers.Dense,
                  tf.keras.layers.DepthwiseConv2D)
        if isinstance(layer, target):
            return tfmot.sparsity.keras.prune_low_magnitude(
                layer, pruning_schedule=schedule)
        return layer

    m = tf.keras.models.load_model(str(MODEL_DIR / "mobilenetv3_fp32.keras"))
    pm = tf.keras.models.clone_model(m, clone_function=apply_pruning)
    pm.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
               loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    pm.fit(train_ds, validation_data=val_ds, epochs=epochs, verbose=verbose,
           callbacks=[tfmot.sparsity.keras.UpdatePruningStep()])

    stripped = tfmot.sparsity.keras.strip_pruning(pm)
    stripped.save(str(out_path))
    print("Saved ->", out_path.name)
    return out_path

for s in PRUNING_LEVELS:
    train_pruned(s)

Epoch 1/3
263/263 [==============================] - 107s 212ms/step - loss: 0.0229 - accuracy: 0.9923 - val_loss: 0.2425 - val_accuracy: 0.9556
Epoch 2/3
263/263 [==============================] - 55s 202ms/step - loss: 0.0225 - accuracy: 0.9929 - val_loss: 0.2232 - val_accuracy: 0.9539
Epoch 3/3
263/263 [==============================] - 55s 202ms/step - loss: 0.0171 - accuracy: 0.9937 - val_loss: 0.2151 - val_accuracy: 0.9528
Saved -> mobilenetv3_pruned_10.keras
Epoch 1/3
263/263 [==============================] - 103s 204ms/step - loss: 0.0206 - accuracy: 0.9919 - val_loss: 0.2664 - val_accuracy: 0.9406
Epoch 2/3
263/263 [==============================] - 54s 199ms/step - loss: 0.0209 - accuracy: 0.9927 - val_loss: 0.1819 - val_accuracy: 0.9744
Epoch 3/3
263/263 [==============================] - 54s 196ms/step - loss: 0.0197 - accuracy: 0.9929 - val_loss: 0.1945 - val_accuracy: 0.9756
Saved -> mobilenetv3_pruned_20.keras
Epoch 1/3
263/263 [==============================] - 103s 20

## Step 9 — Export every variant to TFLite

Produces the folder you copy into the ARM64 VM for the deployability check.

In [ ]:
def export_tflite(keras_path, output_name, quantize_dynamic=False):
    model = tf.keras.models.load_model(str(keras_path))
    conv = tf.lite.TFLiteConverter.from_keras_model(model)
    if quantize_dynamic:
        conv.optimizations = [tf.lite.Optimize.DEFAULT]
    blob = conv.convert()
    out = TFLITE_DIR / output_name
    out.write_bytes(blob)
    print(f"  {output_name:46s} {len(blob)/1e6:6.2f} MB")
    return out

print("FP32 baseline (unquantized):")
export_tflite(MODEL_DIR / "mobilenetv3_fp32.keras", "mobilenetv3_fp32.tflite")

print("\nPruned variants:")
for s in PRUNING_LEVELS:
    pct = int(round(s * 100))
    kp = MODEL_DIR / f"mobilenetv3_pruned_{pct}.keras"
    if not kp.exists():
        print(f"  skip {pct}% — run Step 8 first"); continue
    export_tflite(kp, f"mobilenetv3_pruned_{pct}.tflite")
    export_tflite(kp, f"mobilenetv3_pruned_{pct}_ptq.tflite", quantize_dynamic=True)

print("\nAll models:")
for f in sorted(TFLITE_DIR.glob("*.tflite")):
    print("  ", f.name)

FP32 baseline (unquantized):
  mobilenetv3_fp32.tflite                         11.89 MB

Pruned variants:
  mobilenetv3_pruned_10.tflite                    11.89 MB
  mobilenetv3_pruned_10_ptq.tflite                 3.27 MB
  mobilenetv3_pruned_20.tflite                    11.89 MB
  mobilenetv3_pruned_20_ptq.tflite                 3.27 MB
  mobilenetv3_pruned_30.tflite                    11.89 MB
  mobilenetv3_pruned_30_ptq.tflite                 3.27 MB
  mobilenetv3_pruned_40.tflite                    11.89 MB
  mobilenetv3_pruned_40_ptq.tflite                 3.27 MB
  mobilenetv3_pruned_50.tflite                    11.89 MB
  mobilenetv3_pruned_50_ptq.tflite                 3.27 MB

All models:
   mobilenetv3_fp32.tflite
   mobilenetv3_pruned_10.tflite
   mobilenetv3_pruned_10_ptq.tflite
   mobilenetv3_pruned_20.tflite
   mobilenetv3_pruned_20_ptq.tflite
   mobilenetv3_pruned_30.tflite
   mobilenetv3_pruned_30_ptq.tflite
   mobilenetv3_pruned_40.tflite
   mobilenetv3_pruned_40_ptq

## Step 10 — Per-weather evaluation

In [ ]:
from collections import defaultdict

def eval_per_weather(model_path, is_tflite=False, quiet=False):
    results = defaultdict(lambda: {'correct':0,'total':0})
    test_base = DATA_DIR / "test"
    mean = np.array([123.68, 116.78, 103.94], dtype=np.float32)
    std  = np.float32(58.8)

    if is_tflite:
        interp = tf.lite.Interpreter(
            model_path=str(model_path),
            experimental_op_resolver_type=tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES)
        interp.allocate_tensors()
        inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
        is_int8 = inp['dtype'] == np.int8
    else:
        model = tf.keras.models.load_model(str(model_path))

    for weather in ['sunny','overcast','rainy']:
        for lbl in ['empty','occupied']:
            folder = test_base / weather / lbl
            if not folder.is_dir(): continue
            true_label = 1 if lbl=='occupied' else 0
            for fname in sorted(os.listdir(folder)):
                if not fname.endswith('.jpg'): continue
                img = tf.image.decode_jpeg(tf.io.read_file(str(folder/fname)), channels=3)
                img = tf.image.resize(img,(224,224)).numpy()
                img = (img - mean) / std
                if is_tflite:
                    x = img[None]
                    if is_int8:
                        scale, zp = inp['quantization']
                        x = (x / scale + zp).astype(np.int8)
                    else:
                        x = x.astype(np.float32)
                    interp.set_tensor(inp['index'], x); interp.invoke()
                    pred = int(np.argmax(interp.get_tensor(out['index'])))
                else:
                    pred = int(np.argmax(model(img[None], training=False).numpy()))
                results[weather]['correct'] += int(pred == true_label)
                results[weather]['total']   += 1

    summary, tot_c, tot_n = {}, 0, 0
    for w in ['sunny','overcast','rainy']:
        c, n = results[w]['correct'], results[w]['total']
        if n:
            summary[w] = 100.0*c/n
            if not quiet: print(f"  {w:9s}: {summary[w]:6.2f}%  ({c}/{n})")
        tot_c += c; tot_n += n
    summary['overall'] = 100.0*tot_c/tot_n if tot_n else float('nan')
    if not quiet: print(f"  {'OVERALL':9s}: {summary['overall']:6.2f}%  ({tot_c}/{tot_n})")
    return summary

# print("=== FP32 ===")
# acc_fp32 = eval_per_weather(MODEL_DIR / "mobilenetv3_fp32.keras")
# print("\n=== Dynamic PTQ ===")
# acc_ptq  = eval_per_weather(TFLITE_DIR / "mobilenetv3_ptq_dynamic.tflite", is_tflite=True)

## STEP 1.1 ++

In [ ]:
#CELL 1 - Enviroment Specs

import subprocess, sys, json, platform
import tensorflow as tf

print("=== CPU ===")
print(subprocess.run(["lscpu"], capture_output=True, text=True).stdout)

print("=== MEMORY ===")
print(subprocess.run(["free", "-h"], capture_output=True, text=True).stdout)

print("=== OS ===")
print(open("/etc/os-release").read().split("\n")[0])

print("=== SOFTWARE ===")
print("Python     :", sys.version.split()[0])
print("TensorFlow :", tf.__version__)
print("tf.keras   :", tf.keras.__name__)
print("Platform   :", platform.platform())

gpus = tf.config.list_physical_devices("GPU")
print("GPU        :", gpus)
if gpus:
    print(tf.config.experimental.get_device_details(gpus[0]))

env = {
    "python": sys.version.split()[0],
    "tensorflow": tf.__version__,
    "platform": platform.platform(),
    "gpu": str(gpus),
}
(RESULTS_DIR / "environment.json").write_text(json.dumps(env, indent=2))
print("\nSaved -> results/environment.json")

=== CPU ===
Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  2
On-line CPU(s) list:                     0,1
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.00GHz
CPU family:                              6
Model:                                   85
Thread(s) per core:                      2
Core(s) per socket:                      1
Socket(s):                               1
Stepping:                                3
BogoMIPS:                                4000.30
Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonstop_tsc cpuid tsc

In [ ]:
## QAT training

import json

ABLATION_FILE = RESULTS_DIR / "qat_ablation.json"
qat_ablation = json.loads(ABLATION_FILE.read_text()) if ABLATION_FILE.exists() else []
done = {(round(r["lr"], 8), r["epochs"]) for r in qat_ablation}
print("Already complete:", sorted(done))

for lr in QAT_LEARNING_RATES:
    for ep in QAT_EPOCHS_GRID:
        if (round(lr, 8), ep) in done:
            continue
        print(f"\n=== QAT lr={lr:g} epochs={ep} ===")
        path, hist = train_qat(lr=lr, epochs=ep, verbose=1)
        qat_ablation.append({
            "lr": lr,
            "epochs": ep,
            "tflite": path.name,
            "final_val_acc": float(hist.history["val_accuracy"][-1]),
            "best_val_acc": float(max(hist.history["val_accuracy"])),
        })
        ABLATION_FILE.write_text(json.dumps(qat_ablation, indent=2))
        print(f"  -> val_acc={qat_ablation[-1]['final_val_acc']:.4f} (saved)")

print("\n=== QAT ABLATION SUMMARY ===")
for r in sorted(qat_ablation, key=lambda x: (x["lr"], x["epochs"])):
    print(f"  lr={r['lr']:<8g} epochs={r['epochs']:<3} "
          f"final={r['final_val_acc']*100:6.2f}%  best={r['best_val_acc']*100:6.2f}%")

Already complete: []

=== QAT lr=1e-05 epochs=3 ===
Epoch 1/3
263/263 [==============================] - 145s 458ms/step - loss: 0.7436 - accuracy: 0.6146 - val_loss: 0.7643 - val_accuracy: 0.5133
Epoch 2/3
263/263 [==============================] - 115s 430ms/step - loss: 0.5169 - accuracy: 0.7506 - val_loss: 0.3465 - val_accuracy: 0.8572
Epoch 3/3
263/263 [==============================] - 115s 428ms/step - loss: 0.4174 - accuracy: 0.8165 - val_loss: 0.4482 - val_accuracy: 0.7972


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


QAT -> mobilenetv3_qat_lr1e-05_ep3_bnfrozen.tflite (6.28 MB)
  -> val_acc=0.7972 (saved)

=== QAT lr=1e-05 epochs=10 ===
Epoch 1/10
263/263 [==============================] - 141s 444ms/step - loss: 0.7574 - accuracy: 0.6169 - val_loss: 1.1483 - val_accuracy: 0.5000
Epoch 2/10
263/263 [==============================] - 131s 489ms/step - loss: 0.6046 - accuracy: 0.7267 - val_loss: 0.7076 - val_accuracy: 0.6689
Epoch 3/10
263/263 [==============================] - 128s 467ms/step - loss: 0.4352 - accuracy: 0.7990 - val_loss: 0.2406 - val_accuracy: 0.9172
Epoch 4/10
263/263 [==============================] - 115s 430ms/step - loss: 0.2055 - accuracy: 0.9260 - val_loss: 0.2129 - val_accuracy: 0.9383
Epoch 5/10
263/263 [==============================] - 115s 428ms/step - loss: 0.1662 - accuracy: 0.9445 - val_loss: 0.1975 - val_accuracy: 0.9411
Epoch 6/10
263/263 [==============================] - 115s 425ms/step - loss: 0.1249 - accuracy: 0.9610 - val_loss: 0.1793 - val_accuracy: 0.9422
Epo

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


QAT -> mobilenetv3_qat_lr1e-05_ep10_bnfrozen.tflite (6.28 MB)
  -> val_acc=0.9694 (saved)

=== QAT lr=5e-05 epochs=3 ===
Epoch 1/3
263/263 [==============================] - 137s 433ms/step - loss: 0.7192 - accuracy: 0.6293 - val_loss: 1.2361 - val_accuracy: 0.5000
Epoch 2/3
263/263 [==============================] - 116s 429ms/step - loss: 0.5065 - accuracy: 0.7562 - val_loss: 0.4818 - val_accuracy: 0.7656
Epoch 3/3
263/263 [==============================] - 115s 428ms/step - loss: 0.3437 - accuracy: 0.8631 - val_loss: 0.3137 - val_accuracy: 0.8600


...

QAT -> mobilenetv3_qat_lr5e-05_ep10_bnfrozen.tflite (6.28 MB)
  -> val_acc=0.9683 (saved)

=== QAT lr=0.0001 epochs=3 ===
Epoch 1/3
263/263 [==============================] - 146s 450ms/step - loss: 0.6314 - accuracy: 0.6630 - val_loss: 0.8455 - val_accuracy: 0.5000
Epoch 2/3
263/263 [==============================] - 120s 441ms/step - loss: 0.5971 - accuracy: 0.7235 - val_loss: 0.3962 - val_accuracy: 0.8344
Epoch 3/3
263/263 [==============================] - 119s 442ms/step - loss: 0.3930 - accuracy: 0.8496 - val_loss: 0.2221 - val_accuracy: 0.9344


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


QAT -> mobilenetv3_qat_lr0.0001_ep3_bnfrozen.tflite (6.28 MB)
  -> val_acc=0.9344 (saved)

=== QAT lr=0.0001 epochs=10 ===
Epoch 1/10
263/263 [==============================] - 145s 455ms/step - loss: 0.6653 - accuracy: 0.6510 - val_loss: 0.9499 - val_accuracy: 0.5000
Epoch 2/10
263/263 [==============================] - 119s 437ms/step - loss: 0.5223 - accuracy: 0.7502 - val_loss: 0.4326 - val_accuracy: 0.7583
Epoch 3/10
263/263 [==============================] - 119s 443ms/step - loss: 0.3070 - accuracy: 0.8908 - val_loss: 0.3429 - val_accuracy: 0.8811
Epoch 4/10
263/263 [==============================] - 118s 439ms/step - loss: 0.2934 - accuracy: 0.8944 - val_loss: 0.2117 - val_accuracy: 0.9439
Epoch 5/10
263/263 [==============================] - 118s 437ms/step - loss: 0.2116 - accuracy: 0.9319 - val_loss: 0.2015 - val_accuracy: 0.9417
Epoch 6/10
263/263 [==============================] - 117s 436ms/step - loss: 0.1343 - accuracy: 0.9556 - val_loss: 0.1981 - val_accuracy: 0.9350
E

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


QAT -> mobilenetv3_qat_lr0.0001_ep10_bnfrozen.tflite (6.28 MB)
  -> val_acc=0.9678 (saved)

=== QAT ABLATION SUMMARY ===
  lr=1e-05    epochs=3   final= 79.72%  best= 85.72%
  lr=1e-05    epochs=10  final= 96.94%  best= 96.94%
  lr=5e-05    epochs=3   final= 86.00%  best= 86.00%
  lr=5e-05    epochs=10  final= 96.83%  best= 96.89%
  lr=0.0001   epochs=3   final= 93.44%  best= 93.44%
  lr=0.0001   epochs=10  final= 96.78%  best= 96.94%


In [ ]:
## CELL 3 - Accuracy for every model

import json
import numpy as np
import pandas as pd

ACC_FILE = RESULTS_DIR / "accuracy_all_models.json"
acc_rows = json.loads(ACC_FILE.read_text()) if ACC_FILE.exists() else []
done = {r["model"] for r in acc_rows}

for f in sorted(TFLITE_DIR.glob("*.tflite")):
    if f.name in done:
        print(f"skip {f.name} (done)")
        continue
    print(f"\n--- {f.name} ---")
    s = eval_per_weather(f, is_tflite=True)
    w = [s.get(k, float("nan")) for k in ("sunny", "overcast", "rainy")]
    acc_rows.append({
        "model":        f.name,
        "acc_overall":  round(s["overall"], 2),
        "acc_sunny":    round(w[0], 2),
        "acc_overcast": round(w[1], 2),
        "acc_rainy":    round(w[2], 2),
        "wsi":          round(float(np.std(w, ddof=0)), 3),
    })
    ACC_FILE.write_text(json.dumps(acc_rows, indent=2))

# FP32 Keras baseline is not a .tflite file, so add it separately
if "mobilenetv3_fp32.keras" not in done:
    print("\n--- FP32 baseline (Keras) ---")
    s = eval_per_weather(MODEL_DIR / "mobilenetv3_fp32.keras")
    w = [s.get(k, float("nan")) for k in ("sunny", "overcast", "rainy")]
    acc_rows.append({
        "model": "mobilenetv3_fp32.keras",
        "acc_overall": round(s["overall"], 2),
        "acc_sunny": round(w[0], 2),
        "acc_overcast": round(w[1], 2),
        "acc_rainy": round(w[2], 2),
        "wsi": round(float(np.std(w, ddof=0)), 3),
    })
    ACC_FILE.write_text(json.dumps(acc_rows, indent=2))

# Merge with the efficiency benchmarks from Step 11
bench = json.loads((RESULTS_DIR / "benchmarks_host_cpu.json").read_text())
df = pd.DataFrame(bench).merge(pd.DataFrame(acc_rows), on="model", how="outer")

cols = ["model", "acc_overall", "acc_sunny", "acc_overcast", "acc_rainy", "wsi",
        "size_mb", "latency_ms_per_image", "throughput_fps"]
df = df[[c for c in cols if c in df.columns]].sort_values("model")

df.to_csv(RESULTS_DIR / "full_results.csv", index=False)
print("\nSaved -> results/full_results.csv")
display(df)

skip mobilenetv3_fp32.tflite (done)
skip mobilenetv3_pruned_10.tflite (done)
skip mobilenetv3_pruned_10_ptq.tflite (done)
skip mobilenetv3_pruned_20.tflite (done)
skip mobilenetv3_pruned_20_ptq.tflite (done)
skip mobilenetv3_pruned_30.tflite (done)
skip mobilenetv3_pruned_30_ptq.tflite (done)
skip mobilenetv3_pruned_40.tflite (done)
skip mobilenetv3_pruned_40_ptq.tflite (done)
skip mobilenetv3_pruned_50.tflite (done)
skip mobilenetv3_pruned_50_ptq.tflite (done)
skip mobilenetv3_ptq_dynamic.tflite (done)
skip mobilenetv3_ptq_dynamic_seed0.tflite (done)
skip mobilenetv3_ptq_dynamic_seed1.tflite (done)
skip mobilenetv3_ptq_dynamic_seed2.tflite (done)
skip mobilenetv3_qat_baseline.tflite (done)

--- mobilenetv3_qat_final_int8.tflite ---
  sunny    :  60.83%  (365/600)
  overcast :  64.33%  (386/600)
  rainy    :  70.00%  (420/600)
  OVERALL  :  65.06%  (1171/1800)

--- mobilenetv3_qat_final_noopt.tflite ---
  sunny    :  95.67%  (574/600)
  overcast :  99.17%  (595/600)
  rainy    :  99.00

,model,size_mb,latency_ms_per_image,throughput_fps
0,mobilenetv3_fp32.keras,NaN,NaN,NaN
1,mobilenetv3_fp32.tflite,11.337,9.921,100.80
2,mobilenetv3_pruned_10.tflite,11.337,11.062,90.40
3,mobilenetv3_pruned_10_ptq.tflite,3.116,17.864,55.98
4,mobilenetv3_pruned_20.tflite,11.337,10.937,91.43
5,mobilenetv3_pruned_20_ptq.tflite,3.116,17.399,57.48
6,mobilenetv3_pruned_30.tflite,11.337,10.254,97.53
7,mobilenetv3_pruned_30_ptq.tflite,3.116,17.333,57.69
8,mobilenetv3_pruned_40.tflite,11.337,9.486,105.42
9,mobilenetv3_pruned_40_ptq.tflite,3.116,17.233,58.03


In [ ]:
## Cell 5

import json
import numpy as np

qat_files = sorted(TFLITE_DIR.glob("mobilenetv3_qat_*.tflite"))
rows = []
for f in qat_files:
    print(f"\n--- {f.name} ---")
    s = eval_per_weather(f, is_tflite=True)
    w = [s.get(k, float("nan")) for k in ("sunny", "overcast", "rainy")]
    rows.append({"model": f.name,
                 "test_acc_overall": round(s["overall"], 2),
                 "test_acc_sunny": round(w[0], 2),
                 "test_acc_overcast": round(w[1], 2),
                 "test_acc_rainy": round(w[2], 2),
                 "wsi": round(float(np.std(w, ddof=0)), 3)})

(RESULTS_DIR / "qat_test_accuracy.json").write_text(json.dumps(rows, indent=2))

import pandas as pd
print("\n=== QAT: TEST SET ACCURACY ===")
display(pd.DataFrame(rows).sort_values("test_acc_overall", ascending=False))

In [ ]:
## Cell 4 - McNemar's test

import os, json
import numpy as np
from scipy import stats

def get_predictions(model_path, is_tflite=False):
    """Predictions and labels over the test set, in a fixed deterministic order."""
    preds, trues = [], []
    test_base = DATA_DIR / "test"
    mean = np.array([123.68, 116.78, 103.94], dtype=np.float32)
    std = np.float32(58.8)

    if is_tflite:
        interp = tf.lite.Interpreter(model_path=str(model_path))
        interp.allocate_tensors()
        inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
        is_int8 = inp["dtype"] == np.int8
    else:
        model = tf.keras.models.load_model(str(model_path))

    for weather in ["sunny", "overcast", "rainy"]:
        for lbl in ["empty", "occupied"]:
            folder = test_base / weather / lbl
            if not folder.is_dir():
                continue
            y = 1 if lbl == "occupied" else 0
            for fname in sorted(os.listdir(folder)):
                if not fname.endswith(".jpg"):
                    continue
                img = tf.image.decode_jpeg(tf.io.read_file(str(folder / fname)), channels=3)
                img = (tf.image.resize(img, (224, 224)).numpy() - mean) / std
                if is_tflite:
                    x = img[None]
                    if is_int8:
                        sc, zp = inp["quantization"]
                        x = (x / sc + zp).astype(np.int8)
                    else:
                        x = x.astype(np.float32)
                    interp.set_tensor(inp["index"], x)
                    interp.invoke()
                    p = int(np.argmax(interp.get_tensor(out["index"])))
                else:
                    p = int(np.argmax(model(img[None], training=False).numpy()))
                preds.append(p)
                trues.append(y)
    return np.array(preds), np.array(trues)

# Cache predictions so each model is only evaluated once
_pred_cache = {}
def cached_preds(path, is_tflite):
    key = str(path)
    if key not in _pred_cache:
        _pred_cache[key] = get_predictions(path, is_tflite)
    return _pred_cache[key]

def mcnemar(path_a, path_b, name_a, name_b, a_tflite=False, b_tflite=True):
    pa, y = cached_preds(path_a, a_tflite)
    pb, _ = cached_preds(path_b, b_tflite)
    ca, cb = (pa == y), (pb == y)
    n01 = int(np.sum(ca & ~cb))
    n10 = int(np.sum(~ca & cb))

    print(f"\n{name_a} vs {name_b}   (n={len(y)})")
    print(f"  accuracy: {100*ca.mean():.2f}%  vs  {100*cb.mean():.2f}%")
    print(f"  discordant pairs: {n01} / {n10}")
    if n01 + n10 == 0:
        print("  identical predictions — test not applicable")
        return None
    p = stats.binomtest(n01, n01 + n10, 0.5).pvalue
    print(f"  McNemar exact p = {p:.5f} -> "
          f"{'SIGNIFICANT' if p < 0.05 else 'not significant'} at alpha=0.05")
    return {"a": name_a, "b": name_b,
            "acc_a": float(100*ca.mean()), "acc_b": float(100*cb.mean()),
            "n01": n01, "n10": n10, "p": float(p)}

BASE = MODEL_DIR / "mobilenetv3_fp32.keras"
results = []

comparisons = [
    (TFLITE_DIR / "mobilenetv3_ptq_dynamic.tflite",             "Dynamic PTQ"),
    (TFLITE_DIR / "mobilenetv3_qat_final_int8.tflite",          "QAT INT8"),
    (TFLITE_DIR / "mobilenetv3_qat_final_noopt.tflite",                "QAT no opt"),
    (TFLITE_DIR / "mobilenetv3_pruned_10.tflite",               "Pruned 10%"),
    (TFLITE_DIR / "mobilenetv3_pruned_30.tflite",               "Pruned 30%"),
    (TFLITE_DIR / "mobilenetv3_pruned_50.tflite",               "Pruned 50%"),
    (TFLITE_DIR / "mobilenetv3_pruned_50_ptq.tflite",           "Pruned 50% + PTQ"),
]

for path, label in comparisons:
    if path.exists():
        r = mcnemar(BASE, path, "FP32", label)
        if r:
            results.append(r)
    else:
        print(f"\n(missing: {path.name})")

# Bonferroni correction — several tests against the same baseline
m = len(results)
print(f"\n=== Bonferroni correction (m={m}, alpha_adj={0.05/m:.5f}) ===")
for r in results:
    r["p_bonferroni"] = min(1.0, r["p"] * m)
    print(f"  FP32 vs {r['b']:<18} p={r['p']:.5f}  p_adj={r['p_bonferroni']:.5f}  "
          f"{'SIGNIFICANT' if r['p_bonferroni'] < 0.05 else 'ns'}")

(RESULTS_DIR / "mcnemar.json").write_text(json.dumps(results, indent=2))
print("\nSaved -> results/mcnemar.json")

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:451: UserWarning:     Warning: Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)



FP32 vs Dynamic PTQ   (n=1800)
  accuracy: 96.50%  vs  97.00%
  discordant pairs: 35 / 44
  McNemar exact p = 0.36819 -> not significant at alpha=0.05

FP32 vs QAT INT8   (n=1800)
  accuracy: 96.50%  vs  64.89%
  discordant pairs: 619 / 50
  McNemar exact p = 0.00000 -> SIGNIFICANT at alpha=0.05

FP32 vs QAT no opt   (n=1800)
  accuracy: 96.50%  vs  97.94%
  discordant pairs: 14 / 40
  McNemar exact p = 0.00054 -> SIGNIFICANT at alpha=0.05

FP32 vs Pruned 10%   (n=1800)
  accuracy: 96.50%  vs  96.33%
  discordant pairs: 21 / 18
  McNemar exact p = 0.74926 -> not significant at alpha=0.05

FP32 vs Pruned 30%   (n=1800)
  accuracy: 96.50%  vs  97.17%
  discordant pairs: 25 / 37
  McNemar exact p = 0.16188 -> not significant at alpha=0.05

FP32 vs Pruned 50%   (n=1800)
  accuracy: 96.50%  vs  88.56%
  discordant pairs: 184 / 41
  McNemar exact p = 0.00000 -> SIGNIFICANT at alpha=0.05

FP32 vs Pruned 50% + PTQ   (n=1800)
  accuracy: 96.50%  vs  87.39%
  discordant pairs: 202 / 38
  McNema

In [ ]:
import pandas as pd
df = pd.read_csv(RESULTS_DIR / "benchmarks_host_cpu.csv")
base = df.loc[df.model=="mobilenetv3_fp32.tflite","acc_overall"].iloc[0]
df["acc_degradation_pct"] = ((base - df["acc_overall"]) / base * 100).round(2)
display(df[["model","acc_overall","acc_degradation_pct"]])

,model,acc_overall,acc_degradation_pct
0,mobilenetv3_fp32.tflite,96.50,0.00
1,mobilenetv3_pruned_10.tflite,96.33,0.18
2,mobilenetv3_pruned_10_ptq.tflite,98.17,-1.73
3,mobilenetv3_pruned_20.tflite,98.50,-2.07
4,mobilenetv3_pruned_20_ptq.tflite,98.06,-1.62
5,mobilenetv3_pruned_30.tflite,97.17,-0.69
6,mobilenetv3_pruned_30_ptq.tflite,97.39,-0.92
7,mobilenetv3_pruned_40.tflite,88.44,8.35
8,mobilenetv3_pruned_40_ptq.tflite,91.78,4.89
9,mobilenetv3_pruned_50.tflite,88.56,8.23


In [ ]:
from sklearn.metrics import f1_score
preds, y = cached_preds(TFLITE_DIR/"mobilenetv3_ptq_dynamic.tflite", True)
print(f1_score(y, preds, average="macro"))

0.9699981850753935


## Step 11 — Benchmarks: latency, throughput, size, peak RAM

Reviewer 2 asked for per-image latency, throughput, binary size and peak RAM.

**These are Colab host-CPU numbers.** They are not Raspberry Pi measurements and
must not be labelled as such in the manuscript.

In [ ]:
import time, gc, json, psutil

def benchmark_model(model_path, runs=BENCH_RUNS, warmup=BENCH_WARMUP):
    model_path = str(model_path)
    name = os.path.basename(model_path)
    size_mb = os.path.getsize(model_path) / (1024*1024)

    proc = psutil.Process(os.getpid())
    gc.collect()
    rss_before = proc.memory_info().rss
    dummy = np.random.randn(1, 224, 224, 3).astype(np.float32)

    interp = tf.lite.Interpreter(model_path=model_path)
    interp.allocate_tensors()
    i_idx = interp.get_input_details()[0]['index']
    o_idx = interp.get_output_details()[0]['index']

    for _ in range(warmup):
        interp.set_tensor(i_idx, dummy); interp.invoke()

    peak = proc.memory_info().rss
    t0 = time.perf_counter()
    for _ in range(runs):
        interp.set_tensor(i_idx, dummy)
        interp.invoke()
        _ = interp.get_tensor(o_idx)
        r = proc.memory_info().rss
        if r > peak: peak = r
    t1 = time.perf_counter()

    total = t1 - t0
    return {"model": name,
            "size_mb": round(size_mb, 3),
            "latency_ms_per_image": round(total/runs*1000, 3),
            "throughput_fps": round(runs/total, 2),
            "peak_rss_mb": round(peak/(1024*1024), 1),
            "delta_rss_mb": round((peak-rss_before)/(1024*1024), 1),
            "runs": runs}

rows = []
for f in sorted(TFLITE_DIR.glob("*.tflite")):
    print(f"--- {f.name} ---")
    r = benchmark_model(f)                      # speed / size / memory
    a = eval_per_weather(f, is_tflite=True, quiet=True)   # accuracy

    r["acc_overall"]  = round(a["overall"], 2)
    r["acc_sunny"]    = round(a.get("sunny", float("nan")), 2)
    r["acc_overcast"] = round(a.get("overcast", float("nan")), 2)
    r["acc_rainy"]    = round(a.get("rainy", float("nan")), 2)

    # WSI = standard deviation across weather conditions (lower = more robust)
    w = [a.get(k, float("nan")) for k in ("sunny", "overcast", "rainy")]
    r["wsi"] = round(float(np.std(w, ddof=0)), 3)

    print(f"  acc {r['acc_overall']:.2f}%  |  {r['size_mb']:.2f}MB  "
          f"{r['latency_ms_per_image']:.2f}ms  {r['throughput_fps']:.1f}fps")
    rows.append(r)

(RESULTS_DIR / "benchmarks_host_cpu.json").write_text(json.dumps(rows, indent=2))



import pandas as pd
cols = ["model", "acc_overall", "acc_sunny", "acc_overcast", "acc_rainy", "wsi",
        "size_mb", "latency_ms_per_image", "throughput_fps", "peak_rss_mb"]
df = pd.DataFrame(rows)[cols]
df.to_csv(RESULTS_DIR / "benchmarks_host_cpu.csv", index=False)
print("\nSaved -> results/benchmarks_host_cpu.{json,csv}")
display(df)

--- mobilenetv3_fp32.tflite ---


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:451: UserWarning:     Warning: Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


KeyboardInterrupt: 

In [ ]:
import numpy as np, pandas as pd, json

path = RESULTS_DIR / "benchmarks_host_cpu.csv"
df = pd.read_csv(path)
df["wsi"] = df[["acc_sunny","acc_overcast","acc_rainy"]].std(axis=1, ddof=0).round(3)
df.to_csv(path, index=False)

# keep the JSON in sync so the two don't drift
rows = json.loads((RESULTS_DIR / "benchmarks_host_cpu.json").read_text())
for r in rows:
    w = [r.get(k) for k in ("acc_sunny","acc_overcast","acc_rainy")]
    if all(v is not None for v in w):
        r["wsi"] = round(float(np.std(w, ddof=0)), 3)
(RESULTS_DIR / "benchmarks_host_cpu.json").write_text(json.dumps(rows, indent=2))

print("WSI recomputed as population standard deviation")
display(df[["model","acc_overall","acc_sunny","acc_overcast","acc_rainy","wsi","size_mb",
            "latency_ms_per_image","throughput_fps"]])

WSI recomputed as population standard deviation


,model,acc_overall,acc_sunny,acc_overcast,acc_rainy,wsi,size_mb,latency_ms_per_image,throughput_fps
0,mobilenetv3_fp32.tflite,96.50,93.00,98.33,98.17,2.476,11.337,9.883,101.18
1,mobilenetv3_pruned_10.tflite,96.33,92.50,98.50,98.00,2.718,11.337,9.851,101.51
2,mobilenetv3_pruned_10_ptq.tflite,98.17,95.67,99.33,99.50,1.767,3.116,16.255,61.52
3,mobilenetv3_pruned_20.tflite,98.50,96.00,99.67,99.83,1.769,11.337,9.698,103.12
4,mobilenetv3_pruned_20_ptq.tflite,98.06,96.17,99.17,98.83,1.341,3.116,15.589,64.15
5,mobilenetv3_pruned_30.tflite,97.17,93.83,99.17,98.50,2.375,11.337,9.592,104.26
6,mobilenetv3_pruned_30_ptq.tflite,97.39,95.17,98.67,98.33,1.576,3.116,15.554,64.29
7,mobilenetv3_pruned_40.tflite,88.44,85.00,91.50,88.83,2.668,11.337,9.869,101.33
8,mobilenetv3_pruned_40_ptq.tflite,91.78,89.50,94.33,91.50,1.982,3.116,15.415,64.87
9,mobilenetv3_pruned_50.tflite,88.56,87.33,87.83,90.50,1.392,11.337,9.512,105.13


## Step 12 — Multi-seed runs, confidence intervals, paired t-test

This is the part the paper currently claims but never reports. Slowest step —
retrains the baseline once per seed. Run it last.

In [ ]:
import tensorflow as tf; print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import json, shutil
import numpy as np
from scipy import stats

RESULTS_FILE = RESULTS_DIR / "multi_seed.json"

def multi_seed_study(seeds=SEEDS, resume=True):
    records = []
    if resume and RESULTS_FILE.exists():
        records = json.loads(RESULTS_FILE.read_text())
        done = {r["seed"] for r in records}
        print(f"Resuming — already have seeds: {sorted(done)}")
    else:
        done = set()

    for seed in seeds:
        if seed in done:
            continue
        print(f"\n{'='*40}\nSEED {seed}\n{'='*40}")
        name = f"mobilenetv3_fp32_seed{seed}.keras"
        target = MODEL_DIR / name

        # Seed 0 was already trained as mobilenetv3_fp32.keras — reuse it
        base_existing = MODEL_DIR / "mobilenetv3_fp32.keras"
        if seed == 0 and base_existing.exists() and not target.exists():
            print("Reusing existing seed-0 baseline (skipping ~30 min retrain)")
            shutil.copy(str(base_existing), str(target))
        else:
            build_and_train_fp32(seed=seed, save_name=name, verbose=1)

        ptq_name = f"mobilenetv3_ptq_dynamic_seed{seed}.tflite"
        if not (TFLITE_DIR / ptq_name).exists():
            export_dynamic_ptq(target, ptq_name)

        print(f"\n[seed {seed}] FP32:")
        a_fp32 = eval_per_weather(target)
        print(f"[seed {seed}] PTQ:")
        a_ptq = eval_per_weather(TFLITE_DIR / ptq_name, is_tflite=True)

        records.append({"seed": seed, "fp32": a_fp32, "ptq": a_ptq})
        RESULTS_FILE.write_text(json.dumps(records, indent=2))   # save each seed
        print(f"[seed {seed}] saved")

    return records

records = multi_seed_study()
print(f"\nDone — {len(records)} seeds")

Resuming — already have seeds: [0, 1]

SEED 2
12683000/12683000 [==============================] - 1s 0us/step
Epoch 1/20
263/263 [==============================] - 42s 120ms/step - loss: 0.3785 - accuracy: 0.8415 - val_loss: 0.4664 - val_accuracy: 0.8133
Epoch 2/20
263/263 [==============================] - 34s 111ms/step - loss: 0.3127 - accuracy: 0.8790 - val_loss: 0.3913 - val_accuracy: 0.8783
Epoch 3/20
263/263 [==============================] - 33s 114ms/step - loss: 0.2789 - accuracy: 0.8974 - val_loss: 0.3582 - val_accuracy: 0.8844
Epoch 4/20
263/263 [==============================] - 31s 110ms/step - loss: 0.2585 - accuracy: 0.9056 - val_loss: 0.3388 - val_accuracy: 0.8917
Epoch 5/20
263/263 [==============================] - 32s 114ms/step - loss: 0.2526 - accuracy: 0.9086 - val_loss: 0.3314 - val_accuracy: 0.8961
Epoch 6/20
263/263 [==============================] - 33s 115ms/step - loss: 0.2472 - accuracy: 0.9148 - val_loss: 0.3190 - val_accuracy: 0.9061
Epoch 7/20
263/263 

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:451: UserWarning:     Warning: Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


  sunny    :  91.83%  (551/600)
  overcast :  96.67%  (580/600)
  rainy    :  96.17%  (577/600)
  OVERALL  :  94.89%  (1708/1800)
[seed 2] saved

Done — 3 seeds


In [ ]:
def ci95(a):
    a = np.asarray(a, dtype=float)
    if len(a) < 2:
        return float(a.mean()), float("nan")
    return float(a.mean()), float(stats.sem(a) * stats.t.ppf(0.975, len(a)-1))

out = {"n_seeds": len(records), "per_metric": [], "weather_pairs": []}

# (a) FP32 vs PTQ, paired across seeds
print("=== FP32 vs Dynamic PTQ ===")
for key in ["overall", "sunny", "overcast", "rainy"]:
    f = [r["fp32"][key] for r in records]
    p = [r["ptq"][key]  for r in records]
    mf, hf = ci95(f); mp, hp = ci95(p)
    row = {"metric": key, "fp32_mean": mf, "fp32_ci95": hf,
           "ptq_mean": mp, "ptq_ci95": hp}
    print(f"\n{key.upper()}")
    print(f"  FP32 : {mf:.2f} ± {hf:.2f}")
    print(f"  PTQ  : {mp:.2f} ± {hp:.2f}")
    if len(f) > 1:
        t, pv = stats.ttest_rel(f, p)
        row.update({"t": float(t), "p": float(pv)})
        print(f"  paired t-test: t={t:.4f}, p={pv:.4f} "
              f"({'significant' if pv < 0.05 else 'not significant'} at α=0.05)")
    out["per_metric"].append(row)

# (b) Weather-condition comparisons — what the paper actually claims
print("\n=== Accuracy differences across weather conditions ===")
for model_key in ["fp32", "ptq"]:
    for w1, w2 in [("sunny","overcast"), ("sunny","rainy"), ("overcast","rainy")]:
        a = [r[model_key][w1] for r in records]
        b = [r[model_key][w2] for r in records]
        if len(a) < 2:
            continue
        t, pv = stats.ttest_rel(a, b)
        out["weather_pairs"].append({"model": model_key, "pair": f"{w1}_vs_{w2}",
                                     "mean_diff": float(np.mean(a) - np.mean(b)),
                                     "t": float(t), "p": float(pv)})
        print(f"  {model_key.upper():5s} {w1} vs {w2}: "
              f"Δ={np.mean(a)-np.mean(b):+.2f}, t={t:.3f}, p={pv:.4f}")

# WSI = standard deviation across weather conditions (matches your paper's definition)
for model_key in ["fp32", "ptq"]:
    wsis = [float(np.std([r[model_key][w] for w in ("sunny","overcast","rainy")], ddof=0))
            for r in records]
    m, h = ci95(wsis)
    out[f"wsi_{model_key}"] = {"mean": m, "ci95": h}
    print(f"\nWSI ({model_key.upper()}): {m:.3f} ± {h:.3f}")

(RESULTS_DIR / "significance.json").write_text(json.dumps(out, indent=2))
print("\nSaved -> results/significance.json")

=== FP32 vs Dynamic PTQ ===

OVERALL
  FP32 : 97.63 ± 2.44
  PTQ  : 96.57 ± 3.96
  paired t-test: t=0.9658, p=0.4360 (not significant at α=0.05)

SUNNY
  FP32 : 95.11 ± 4.54
  PTQ  : 94.28 ± 5.40
  paired t-test: t=0.4484, p=0.6978 (not significant at α=0.05)

OVERCAST
  FP32 : 98.89 ± 1.45
  PTQ  : 98.00 ± 3.31
  paired t-test: t=1.3874, p=0.2997 (not significant at α=0.05)

RAINY
  FP32 : 98.89 ± 1.57
  PTQ  : 97.44 ± 3.32
  paired t-test: t=1.6543, p=0.2399 (not significant at α=0.05)

=== Accuracy differences across weather conditions ===
  FP32  sunny vs overcast: Δ=-3.78, t=-4.715, p=0.0422
  FP32  sunny vs rainy: Δ=-3.78, t=-5.427, p=0.0323
  FP32  overcast vs rainy: Δ=-0.00, t=-0.000, p=1.0000
  PTQ   sunny vs overcast: Δ=-3.72, t=-6.602, p=0.0222
  PTQ   sunny vs rainy: Δ=-3.17, t=-5.270, p=0.0342
  PTQ   overcast vs rainy: Δ=+0.56, t=10.000, p=0.0099

WSI (FP32): 1.788 ± 1.485

WSI (PTQ): 1.641 ± 1.160

Saved -> results/significance.json


## Step 13 — Package TFLite models for the ARM64 VM

In [ ]:
import shutil
shutil.make_archive('/content/tflite_models', 'zip', str(TFLITE_DIR))
print("Created /content/tflite_models.zip")

from google.colab import files
files.download('/content/tflite_models.zip')

Created /content/tflite_models.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>